<a href="https://colab.research.google.com/github/taek20230541/maritimedatamining/blob/main/Colab_%EC%8B%9C%EC%9E%91%ED%95%98%EA%B8%B0.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 11주차 문제1

In [4]:
import pandas as pd
import numpy as np
from scipy.stats import chi2_contingency
import os

# 1. 데이터 불러오기 (경로 에러 방지)
file_path = 'channel_purchase.csv' # 파일이 같은 폴더에 있다고 가정

if os.path.exists(file_path):
    df = pd.read_csv(file_path)
    print("✅ 파일을 성공적으로 로드했습니다.")
else:
    print("⚠️ 파일을 찾을 수 없어 실습용 샘플 데이터를 생성합니다.")
    # 실습용 가상 데이터 생성
    data = {
        'channel': ['Direct']*40 + ['Search']*40 + ['SNS']*60 + ['Email']*60,
        'purchase_yn': [1]*30 + [0]*10 + [1]*25 + [0]*15 + [1]*30 + [0]*30 + [1]*20 + [0]*40
    }
    df = pd.DataFrame(data)

# 2. 데이터 전처리 (에러 방지 핵심)
# purchase_yn이 'Yes'/'No' 같은 문자열일 경우를 대비해 0과 1로 변환
if df['purchase_yn'].dtype == 'object':
    # 구매를 뜻하는 단어들을 1로, 나머지를 0으로 매핑
    mapping = {'Yes': 1, 'No': 0, '구매': 1, '미구매': 0, 'Y': 1, 'N': 0}
    df['purchase_yn_num'] = df['purchase_yn'].map(mapping).fillna(0)
else:
    df['purchase_yn_num'] = df['purchase_yn'].astype(int)

# 3. 교차표 생성
# 카이제곱 검정에는 원래의 범주형(또는 정수형) 데이터가 들어간 table을 사용합니다.
table = pd.crosstab(df['channel'], df['purchase_yn'])

print("\n[1] 관측도수 (교차표)")
print(table)

# 4. 카이제곱 검정 (chi2_contingency)
chi2, p, dof, expected = chi2_contingency(table)

print("\n[2] 검정 결과")
print(f"- 카이제곱 통계량: {chi2:.4f}")
print(f"- p-value: {p:.6f}")
print(f"- 자유도: {dof}")

# 5. 채널별 구매전환율 계산 (해석용)
# numeric_only=True 옵션을 주거나 명시적으로 수치 컬럼을 지정하여 TypeError 방지
conv_rate = df.groupby('channel')['purchase_yn_num'].mean().sort_values(ascending=False) * 100

print("\n[3] 채널별 구매전환율(%)")
print(conv_rate.round(2))

# 6. 최종 해석
print("\n[4] 통계적 해석")
alpha = 0.05
if p < alpha:
    print(f"결론: p-value가 {alpha}보다 작으므로, 유입 채널과 구매 여부 사이에는 유의미한 관련이 있습니다.")
    print(f"👉 성과가 가장 좋은 채널은 '{conv_rate.index[0]}'입니다.")
else:
    print(f"결론: p-value가 {alpha}보다 크므로, 채널과 구매 여부는 서로 독립적(관련 없음)입니다.")

⚠️ 파일을 찾을 수 없어 실습용 샘플 데이터를 생성합니다.

[1] 관측도수 (교차표)
purchase_yn   0   1
channel            
Direct       10  30
Email        40  20
SNS          30  30
Search       15  25

[2] 검정 결과
- 카이제곱 통계량: 18.7135
- p-value: 0.000313
- 자유도: 3

[3] 채널별 구매전환율(%)
channel
Direct    75.00
Search    62.50
SNS       50.00
Email     33.33
Name: purchase_yn_num, dtype: float64

[4] 통계적 해석
결론: p-value가 0.05보다 작으므로, 유입 채널과 구매 여부 사이에는 유의미한 관련이 있습니다.
👉 성과가 가장 좋은 채널은 'Direct'입니다.


# 11주차 문제 2

In [5]:
import pandas as pd
import numpy as np
from scipy.stats import chi2_contingency
import os

# ---------------------------------------------------------
# 1. 데이터 로드 (경로 에러 방지 처리)
# ---------------------------------------------------------
file_path = 'gender_preference.csv'

if os.path.exists(file_path):
    df = pd.read_csv(file_path)
    print("✅ 데이터를 성공적으로 로드했습니다.")
else:
    print("⚠️ 파일을 찾을 수 없어 샘플 데이터를 생성합니다.")
    # 실제 파일이 없을 경우를 대비한 강의용 샘플 데이터
    data = {
        'gender': ['Male']*100 + ['Female']*100,
        'content_type': (['Programming']*60 + ['Marketing']*20 + ['Design']*20) +
                        (['Programming']*20 + ['Marketing']*50 + ['Design']*30)
    }
    df = pd.DataFrame(data)

# ---------------------------------------------------------
# 2. 교차표(Contingency Table) 및 비중 분석
# ---------------------------------------------------------
# 빈도 확인
table = pd.crosstab(df['gender'], df['content_type'])

# 성별 내 비율(%) 확인 - 동질성 검정에서 매우 중요!
row_ratio = pd.crosstab(df['gender'], df['content_type'], normalize='index') * 100

print("\n[1] 성별별 콘텐츠 선호 빈도 (관측도수)")
print(table)
print("\n[2] 성별 내 콘텐츠 선호 비중 (%)")
print(row_ratio.round(2))

# ---------------------------------------------------------
# 3. 카이제곱 동질성 검정 수행
# ---------------------------------------------------------
chi2, p, dof, expected = chi2_contingency(table)

print("\n[3] 카이제곱 검정 결과")
print(f"- 카이제곱 통계량: {chi2:.4f}")
print(f"- p-value: {p:.6f}")
print(f"- 자유도: {dof}")

# ---------------------------------------------------------
# 4. 심층 해석 지표 (최다 선호 및 기대도수)
# ---------------------------------------------------------
# 각 성별별로 어떤 콘텐츠를 가장 선호하는가?
top_pref = table.idxmax(axis=1)

print("\n[4] 성별별 최다 선호 콘텐츠")
for gender, content in top_pref.items():
    print(f"- {gender:6s}: {content}")

# ---------------------------------------------------------
# 5. 최종 결과 보고
# ---------------------------------------------------------
alpha = 0.05
print("\n" + "="*50)
print("📌 최종 분석 리포트")
print("-"*50)
if p < alpha:
    print(f"결론: p-value({p:.6f}) < {alpha}")
    print("해석: 성별에 따른 콘텐츠 선호 분포는 통계적으로 유의미하게 '다릅니다'.")
    print("      즉, 남성과 여성의 선호도는 동질하지 않습니다.")
else:
    print(f"결론: p-value({p:.6f}) >= {alpha}")
    print("해석: 성별에 따른 콘텐츠 선호 분포는 통계적으로 '동질합니다'.")
    print("      성별에 따른 선호 차이가 있다고 보기 어렵습니다.")
print("="*50)

⚠️ 파일을 찾을 수 없어 샘플 데이터를 생성합니다.

[1] 성별별 콘텐츠 선호 빈도 (관측도수)
content_type  Design  Marketing  Programming
gender                                      
Female            30         50           20
Male              20         20           60

[2] 성별 내 콘텐츠 선호 비중 (%)
content_type  Design  Marketing  Programming
gender                                      
Female          30.0       50.0         20.0
Male            20.0       20.0         60.0

[3] 카이제곱 검정 결과
- 카이제곱 통계량: 34.8571
- p-value: 0.000000
- 자유도: 2

[4] 성별별 최다 선호 콘텐츠
- Female: Marketing
- Male  : Programming

📌 최종 분석 리포트
--------------------------------------------------
결론: p-value(0.000000) < 0.05
해석: 성별에 따른 콘텐츠 선호 분포는 통계적으로 유의미하게 '다릅니다'.
      즉, 남성과 여성의 선호도는 동질하지 않습니다.
